In [1]:
# Run once (needs internet). Remove "-q" if you want to see the install log.
%pip install -q numpy scikit-learn sentence-transformers

In [2]:
import re

samples = [
    "AI, search!",                 # punctuation
    "I can't log in.",             # contraction
    "help@example.com",            # email
    "CS4944",                      # identifier
    "",                            # empty string
    "یہ ایک مثال ہے",               # Urdu text
    "2.1 version 4944CS ",         # numbers, identifier, trailing space
]

rows = []
for text in samples:
    ws = text.split()
    words = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
    rows.append((text, ws, words))
    print("Original  :", repr(text))
    print("Whitespace:", ws, len(ws))
    print("Word-like :", words, len(words))
    print("-" * 60)

assert re.findall(r"\b\w+\b", "") == []

Original  : 'AI, search!'
Whitespace: ['AI,', 'search!'] 2
Word-like : ['AI', 'search'] 2
------------------------------------------------------------
Original  : "I can't log in."
Whitespace: ['I', "can't", 'log', 'in.'] 4
Word-like : ['I', 'can', 't', 'log', 'in'] 5
------------------------------------------------------------
Original  : 'help@example.com'
Whitespace: ['help@example.com'] 1
Word-like : ['help', 'example', 'com'] 3
------------------------------------------------------------
Original  : 'CS4944'
Whitespace: ['CS4944'] 1
Word-like : ['CS4944'] 1
------------------------------------------------------------
Original  : ''
Whitespace: [] 0
Word-like : [] 0
------------------------------------------------------------
Original  : 'یہ ایک مثال ہے'
Whitespace: ['یہ', 'ایک', 'مثال', 'ہے'] 4
Word-like : ['یہ', 'ایک', 'مثال', 'ہے'] 4
------------------------------------------------------------
Original  : '2.1 version 4944CS '
Whitespace: ['2.1', 'version', '4944CS'] 3
Word-like

In [3]:
# Optional: needs the model download. If it fails, record the limitation in the report.
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    for text in ["AI, search!", "I can't log in."]:
        toks = model.tokenizer.tokenize(text)
        print(repr(text), "->", toks, len(toks), "word pieces (without special tokens)")
except Exception as e:
    model = None
    print("Model tokenizer not available:", type(e).__name__, e)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

'AI, search!' -> ['ai', ',', 'search', '!'] 4 word pieces (without special tokens)
"I can't log in." -> ['i', 'can', "'", 't', 'log', 'in', '.'] 7 word pieces (without special tokens)


In [4]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


def load_encoder():
    # First call downloads the model (needs internet); later runs use the cache.
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(MODEL_NAME, device="cpu")


class Retriever:
    """Same behaviour as lab6_search.Retriever; an already-loaded encoder can be passed in."""

    def __init__(self, records, backend="tfidf", model=None):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)
        elif backend == "embedding":
            self.model = model or load_encoder()
            self.check_length(texts)
            self.matrix = self.model.encode(texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")

    def check_length(self, texts):
        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)["input_ids"])
            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")

    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        if self.backend == "tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz == 0:
                return []
            scores = (q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q = self.model.encode([query], normalize_embeddings=True)[0]
            scores = self.matrix @ q
        allowed = [i for i, r in enumerate(self.records)
                   if all(r.get(key) == value for key, value in (filters or {}).items())]
        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in ranked[:k]]


def show_hits(hits):
    if not hits:
        print("No matching records.")
    for record, score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])

In [5]:
FAQ = [
    {"id": "faq:password", "question": "How do I reset my password?",
     "answer": "Use the recovery link on the login page.",
     "source": "it_help.txt"},
    {"id": "faq:submit", "question": "Where do I submit my assignment?",
     "answer": "Use the assignment upload page in the learning portal.",
     "source": "assessment.txt"},
    {"id": "faq:late", "question": "Can illness justify a late submission?",
     "answer": "Request an extension with medical evidence before the deadline.",
     "source": "handbook.txt"},
    # ---- two FAQs added for Task 1 (synthetic, each with a source) ----
    {"id": "faq:wifi", "question": "How do I connect to the campus Wi-Fi?",
     "answer": "Select the university network and sign in with your student account.",
     "source": "it_help.txt"},
    {"id": "faq:fees", "question": "Where can I pay my semester fee?",
     "answer": "Pay the semester fee at the accounts office or through the student portal.",
     "source": "fee_rules.txt"},
]


def build_bot(backend="tfidf"):
    records = [dict(r, text=r["question"] + " " + r["answer"]) for r in FAQ]
    return Retriever(records, backend)


def reply(engine, query):
    if not query.strip():
        return "Please enter a question."
    # Only exact canonical FAQ text is answered automatically here.
    for r in FAQ:
        if query.strip().casefold() == r["question"].casefold():
            return r["answer"] + " [source: " + r["source"] + "]"
    hits = engine.search(query)
    if not hits:
        return "No matching FAQ. Please contact the help desk."
    lines = ["Possible matches; choose or rephrase your question:"]
    for r, score in hits:
        lines.append(r["question"] + f" (similarity {score:.3f})")
    return "\n".join(lines)


engine = build_bot("tfidf")
print("FAQ bot ready with", len(FAQ), "FAQs")

FAQ bot ready with 5 FAQs


In [6]:
tests = [
    ("exact answer (original FAQ)",   "How do I reset my password?"),
    ("exact answer (added FAQ)",      "How do I connect to the campus Wi-Fi?"),
    ("suggestion (non-exact)",        "password recovery"),
    ("suggestion for added FAQ",      "pay fee"),
    ("empty input",                   ""),
    ("unsupported input",             "volcano geology"),
]
for label, query in tests:
    print(f"[{label}]")
    print("You:", repr(query))
    print("Bot:", reply(engine, query))
    print()

[exact answer (original FAQ)]
You: 'How do I reset my password?'
Bot: Use the recovery link on the login page. [source: it_help.txt]

[exact answer (added FAQ)]
You: 'How do I connect to the campus Wi-Fi?'
Bot: Select the university network and sign in with your student account. [source: it_help.txt]

[suggestion (non-exact)]
You: 'password recovery'
Bot: Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.453)
Where do I submit my assignment? (similarity 0.000)
Can illness justify a late submission? (similarity 0.000)

[suggestion for added FAQ]
You: 'pay fee'
Bot: Possible matches; choose or rephrase your question:
Where can I pay my semester fee? (similarity 0.602)
How do I reset my password? (similarity 0.000)
Where do I submit my assignment? (similarity 0.000)

[empty input]
You: ''
Bot: Please enter a question.

[unsupported input]
You: 'volcano geology'
Bot: No matching FAQ. Please contact the help desk.



In [7]:
# Automatic behaviour checks (same idea as task1_chatbot_check.py)
assert "it_help.txt" in reply(engine, "How do I reset my password?")
assert reply(engine, "") == "Please enter a question."
assert "No matching FAQ" in reply(engine, "volcano geology")
assert "Possible matches" in reply(engine, "password recovery")
print("FAQ behaviour checks passed")

FAQ behaviour checks passed


In [8]:
print("FAQ assistant. Type exit to stop.")
while True:
    try:
        query = input("You: ")
    except (EOFError, KeyboardInterrupt):
        break
    if query.strip().lower() == "exit":
        break
    print("Bot:", reply(engine, query))

FAQ assistant. Type exit to stop.
